# Bài thực hành Deep Learning: Recurrent Neural Network

Notebook này cài đặt RNN để dự báo chuỗi thời gian cho 4 bài: giá nhà, giá Bitcoin, điện thế tiêu thụ và giá đóng cửa NIFTY.

## 1. Mục tiêu bài thực hành

- Đọc và xử lý dữ liệu chuỗi thời gian.
- Chuẩn hóa dữ liệu bằng `MinMaxScaler`.
- Tạo dữ liệu dạng RNN với `X shape = samples, time_steps, features`.
- Xây dựng mô hình `SimpleRNN` để dự báo giá trị tiếp theo.
- Đánh giá mô hình bằng RMSE và MAE.
- Lưu model, scaler và thử dự báo giá trị mới.

## 2. Giới thiệu RNN

RNN là mạng nơ-ron hồi tiếp dùng để xử lý dữ liệu tuần tự. Ở mỗi bước thời gian, RNN nhận đầu vào hiện tại và thông tin từ các bước trước, nhờ đó có thể học quy luật biến động trong chuỗi thời gian.

## 3. RNN dùng cho bài toán chuỗi thời gian

Với dự báo chuỗi thời gian, ta lấy `time_steps` giá trị gần nhất làm đầu vào để dự báo giá trị tiếp theo. Ví dụ, với `time_steps = 12`, mô hình dùng 12 giá trị quá khứ để dự báo giá trị thứ 13.

## 4. Import thư viện

In [ ]:
# Nếu chạy trên Colab và thiếu thư viện, mở comment dòng dưới:
# !pip install tensorflow numpy pandas matplotlib scikit-learn flask joblib yfinance requests

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT_DIR = Path.cwd()
if not (ROOT_DIR / 'utils').exists() and (ROOT_DIR / 'rnn_deeplearning_project').exists():
    ROOT_DIR = ROOT_DIR / 'rnn_deeplearning_project'
sys.path.append(str(ROOT_DIR))

from utils.data_loader import DATASET_CONFIG, ensure_dataset, get_dataset_config, get_target_series, normalize_target_column
from utils.preprocessing import train_rnn_for_dataset
from utils.prediction import predict_next_value

TIME_STEPS = 12
EPOCHS = 20

## 5. Hàm dùng chung

Các hàm dưới đây dùng để load dữ liệu, chuẩn hóa, tạo sequence, xây dựng/train model, đánh giá model và vẽ biểu đồ. Phần xử lý chính được đặt trong thư mục `utils/` để notebook và Flask app dùng lại cùng một logic.

In [ ]:
def hien_thi_du_lieu(dataset_key):
    """Hiển thị 5 dòng đầu và biểu đồ chuỗi thời gian ban đầu."""
    config = get_dataset_config(dataset_key)
    df = ensure_dataset(dataset_key)
    df = normalize_target_column(df, config['target'])
    display(df.head())

    series = pd.to_numeric(df[config['target']], errors='coerce').ffill().bfill().dropna()
    plt.figure(figsize=(10, 4))
    plt.plot(series.values)
    plt.title(f"Chuỗi thời gian ban đầu - {config['display_name']}")
    plt.xlabel('Thời điểm')
    plt.ylabel(config['target'])
    plt.tight_layout()
    plt.show()
    return series


def ve_loss(history, title):
    """Vẽ biểu đồ loss và val_loss."""
    plt.figure(figsize=(8, 4))
    plt.plot(history.history['loss'], label='loss')
    plt.plot(history.history['val_loss'], label='val_loss')
    plt.title(f'Loss/Val_loss - {title}')
    plt.xlabel('Epoch')
    plt.ylabel('MSE')
    plt.legend()
    plt.tight_layout()
    plt.show()


def ve_actual_prediction(actual, prediction, title):
    """Vẽ biểu đồ Actual vs Prediction."""
    plt.figure(figsize=(10, 4))
    plt.plot(actual, label='Actual')
    plt.plot(prediction, label='Prediction')
    plt.title(f'Actual vs Prediction - {title}')
    plt.xlabel('Mẫu test')
    plt.ylabel('Giá trị thật')
    plt.legend()
    plt.tight_layout()
    plt.show()


def chay_bai_thuc_hanh(dataset_key, epochs=EPOCHS, time_steps=TIME_STEPS):
    """Chạy trọn pipeline: xem dữ liệu, train, đánh giá, vẽ biểu đồ."""
    config = get_dataset_config(dataset_key)
    print(f"\n===== {config['display_name']} =====")
    hien_thi_du_lieu(dataset_key)
    result = train_rnn_for_dataset(
        dataset_key,
        time_steps=time_steps,
        epochs=epochs,
        batch_size=16,
        save_plots=False,
        verbose=1,
    )
    ve_loss(result['history'], config['display_name'])
    ve_actual_prediction(result['actual'], result['prediction'], config['display_name'])
    print(f"RMSE: {result['rmse']:.4f}")
    print(f"MAE : {result['mae']:.4f}")
    print(f"Model đã lưu: {result['model_path']}")
    return result

## 6. Bài 1: Dự báo giá nhà `price`

Dataset cần có file `datasets/raw_sales.csv` và cột dự báo tên `price`. Nếu chưa có file, code sẽ thử tải dataset giá nhà public tương đương và đổi tên cột giá trị nhà thành `price`.

In [ ]:
result_house = chay_bai_thuc_hanh('house')

## 7. Bài 2: Dự báo giá Bitcoin `priceUSD`

Dataset `datasets/BTC_DATA.csv` được lấy từ `BTC-USD` trên `yfinance`, cột `Close` được đổi tên thành `priceUSD`.

In [ ]:
result_btc = chay_bai_thuc_hanh('btc')

## 8. Bài 3: Dự báo điện thế `Voltage`

Dataset `datasets/household_power_consumption.txt` dùng nguồn Individual Household Electric Power Consumption. Missing value ký hiệu `?` được chuyển thành NaN rồi xử lý trước khi train.

In [ ]:
result_voltage = chay_bai_thuc_hanh('voltage')

## 9. Bài 4: Dự báo giá đóng cửa NIFTY `close`

Dataset `datasets/NIFTY_stock_market.csv` được lấy từ mã `^NSEI` trên `yfinance`, cột `Close` được đổi tên thành `close`.

In [ ]:
result_nifty = chay_bai_thuc_hanh('nifty')

## 10. So sánh kết quả

In [ ]:
results = [result_house, result_btc, result_voltage, result_nifty]
summary = pd.DataFrame([
    {
        'Tên bài': item['title'],
        'Dataset': item['dataset'],
        'Cột dự báo': item['target'],
        'Time steps': item['time_steps'],
        'Model': item['model'],
        'RMSE': item['rmse'],
        'MAE': item['mae'],
        'File model đã lưu': item['model_path'],
    }
    for item in results
])
display(summary)

## 11. Lưu mô hình

Sau khi train, các model được lưu trong thư mục `models/`:

- `models/rnn_house_price.h5`
- `models/rnn_btc_price.h5`
- `models/rnn_voltage.h5`
- `models/rnn_nifty_close.h5`

Các scaler được lưu trong thư mục `scalers/` bằng `joblib`.

## 12. Dự báo giá trị mới

Hàm `predict_next_value` nhận đúng `time_steps` giá trị gần nhất, chuẩn hóa input, reshape thành `(1, time_steps, 1)`, dự báo bằng model đã lưu, rồi đưa kết quả về giá trị thật bằng `inverse_transform`.

In [ ]:
# Ví dụ: dùng 12 giá trị Bitcoin gần nhất để dự báo giá trị tiếp theo.
recent_btc_values = get_target_series('btc').tail(TIME_STEPS).tolist()
next_btc_price = predict_next_value('btc', recent_btc_values, time_steps=TIME_STEPS)
print('12 giá trị gần nhất:', recent_btc_values)
print(f'Giá trị BTC dự báo tiếp theo: {next_btc_price:.4f}')

## 13. Nhận xét

- RNN phù hợp với dữ liệu chuỗi thời gian vì mô hình có thể dùng thông tin từ các bước thời gian trước.
- `time_steps` càng lớn thì mô hình có nhiều thông tin quá khứ hơn nhưng train lâu hơn.
- Dữ liệu tài chính như BTC và NIFTY biến động mạnh nên dự báo khó hơn dữ liệu có quy luật ổn định.
- Dữ liệu thiếu cần xử lý trước khi train, đặc biệt dataset điện năng có missing value ký hiệu `?`.
- Kết quả dự báo chỉ dùng cho mục đích học tập, không dùng để đầu tư thực tế.

## 14. Kết luận

Notebook đã hoàn thành 4 bài dự báo chuỗi thời gian bằng SimpleRNN, có chuẩn hóa dữ liệu, chia train/test, đánh giá bằng RMSE/MAE, vẽ biểu đồ loss và Actual vs Prediction, lưu model/scaler và dự báo giá trị tiếp theo.